# Split the knocked-down genes into folds

Each knocked-down gene goes into one of five folds, the same fold in every context of its dataset.
Folds 0–3 are for development, and fold 4 is held out until the end.
For development fold `f`, the test set is fold `f` and the validation set is 20% of the other development folds' genes.
PIE's benchmark keeps PIE's published splits.

The knockdowns of each dataset were chosen on the cluster before any fold was drawn, and `splits/<dataset>.tsv` lists them, one row per context and knockdown:
TxPert's knockdown sets for Replogle K562, RPE1 and Nadig HepG2, Jurkat, and Marson's rule (more than 75 DE genes and more than 50 cells) for the others.
For the two mouse datasets a knockout also needs a one-to-one human orthologue, and genes are named by it.

This notebook draws the folds and splits from those lists and rewrites the `fold` and `split_f0`–`split_f3` columns, which git keeps.
It reproduces the cluster's folds and the splits PIE and PWPA were trained on exactly.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
for path in sorted(Path("splits").glob("*.tsv")):
    rows = pd.read_csv(path, sep="\t")

    # five folds over the sorted genes (human ENSG, also for the mouse datasets), the same fold in every context
    genes = np.sort(rows.ensg.unique())
    fold = np.empty(len(genes), int)
    fold[np.random.default_rng(0).permutation(len(genes))] = np.arange(len(genes)) % 5
    rows["fold"] = rows.ensg.map(dict(zip(genes, fold)))

    # development fold f: test = fold f, validation = 20% of the other development folds' genes, train = the rest
    for f in range(4):
        dev = np.sort(rows.symbol[(rows.fold != f) & (rows.fold != 4)].unique())
        val = set(np.random.default_rng(0).permutation(dev)[: round(0.2 * len(dev))])
        rows[f"split_f{f}"] = np.select([rows.fold == 4, rows.fold == f, rows.symbol.isin(val)], ["lockbox", "test", "val"], "train")

    rows.to_csv(path, sep="\t", index=False)
    print(path.stem, len(rows), "rows,", len(genes), "genes; fold sizes", np.bincount(fold).tolist(), "; f0", rows.split_f0.value_counts().to_dict())